# CNN Architecture Course: Sequential API, Functional API, and Classic Building Blocks



## 1. The Sequential API

The `Sequential` model is a straight stack of layers: input flows through layer 1, then layer 2, and so on, with no branching or skipping allowed. It's the right tool when your architecture really is a single chain.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, BatchNormalization, Add, Input

# Create a new Sequential model
model = Sequential([Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
                BatchNormalization(),
                MaxPooling2D(pool_size=(2, 2)),
                Conv2D(64, (3, 3), activation='relu'),
                Flatten(),
                Dense(128, activation='relu'),
                Dense(4, activation='softmax')])


# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the summary of the model
model.summary()


## 2. The Functional API

The `Functional` API treats each layer as a callable applied to a tensor, and you assemble the model by explicitly wiring tensors together with `Model(inputs=..., outputs=...)`. This is what you need the moment an architecture is not a single chain: multiple inputs/outputs, shared layers, or — as in the next two sections — branches that split and rejoin.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, MaxPooling2D, Flatten, Dense, Activation
from tensorflow.keras.models import Model

# Define the input shape
input_shape = (224, 224, 3)

# Define the input tensor
inputs = Input(shape=input_shape)

# Add a Conv2D layer with 32 filters, a 3x3 kernel, and 'relu' activation
x = Conv2D(32, (3, 3), activation='relu')(inputs)
x = BatchNormalization()(x)

# Add a MaxPooling2D layer with 2x2 pool size
x = MaxPooling2D(pool_size=(2, 2))(x)

# Add another Conv2D layer with 64 filters, a 3x3 kernel, and 'relu' activation
x = Conv2D(64, (3, 3), activation='relu')(x)
x = BatchNormalization()(x)

# Flatten the feature maps
x = Flatten()(x)

# Add a fully connected Dense layer with 128 units and 'relu' activation
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)

# Add the output layer with the desired number of units and activation function
outputs = Dense(4, activation='softmax')(x)

# Create the model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the summary of the model
model.summary()


**Note on parameter counts:** the Sequential and Functional versions above have essentially the same trainable parameters (~97.3M, dominated by the first Dense layer after `Flatten`). The Functional version adds one extra `BatchNormalization` after the first Dense layer, which is why its total is slightly higher. The point of this comparison isn't parameter count — it's that both APIs describe the *same chain* equally well. The gap only shows up once the architecture stops being a chain, which is exactly what happens next.

## 3. Residual (ResNet-style) Block

Reference figure: [He et al., *Deep Residual Learning for Image Recognition*](https://arxiv.org/abs/1512.03385), Figure 4.

A residual block adds the block's input back to its output (`Add()([x, residual])`) before the final activation. This skip connection is precisely the kind of non-chain wiring the Sequential API cannot express — you *need* the Functional API here.

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Add, Activation, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

# Define the input shape
input_shape = (224, 224, 3)

# Define the input tensor
inputs = Input(shape=input_shape)

# First Convolutional Block
x = Conv2D(64, (7, 7), strides=(2, 2), padding='same', activation='relu')(inputs)
x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

# Residual Block
residual = x

x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
x = Conv2D(64, (3, 3), padding='same')(x)
x = Add()([x, residual])
x = Activation('relu')(x)


# Average Pooling
x = GlobalAveragePooling2D()(x)

# Output layer
outputs = Dense(4, activation='softmax')(x)

# Create the model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the summary of the model
model.summary()


## 4. Implement the Inception Block

Reference: [Szegedy et al., *Going Deeper with Convolutions*](https://arxiv.org/abs/1409.4842) (GoogLeNet), the "naive" Inception module (Figure 2a).

**Idea:** instead of choosing one kernel size, run several convolutions of *different* kernel sizes (1x1, 3x3, 5x5) plus a pooling branch **in parallel** on the same input, then concatenate their outputs along the channel axis. This lets the network capture features at multiple scales simultaneously.

1x1 convolutions are inserted before the 3x3/5x5 branches (and after the pool branch) purely as *bottlenecks* — they reduce the channel depth before the more expensive convolutions run, which is what keeps the real GoogLeNet computationally tractable (this is the "dimension reduction" version of the module; the version below shows both the naive and reduced forms).

This is the third and last reason you need the Functional API in this notebook: parallel branches that share one input and rejoin via `Concatenate`, which — like the residual `Add` above — cannot be expressed as a single Sequential chain.

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Concatenate, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model


def inception_module(x, filters_1x1, filters_3x3_reduce, filters_3x3,
                      filters_5x5_reduce, filters_5x5, filters_pool_proj, name='inception'):
    """GoogLeNet-style Inception module with dimensionality-reduction bottlenecks.

    Four parallel branches on the same input `x`, concatenated on the channel axis:
      1) 1x1 conv
      2) 1x1 conv (reduce) -> 3x3 conv
      3) 1x1 conv (reduce) -> 5x5 conv
      4) 3x3 max-pool -> 1x1 conv (projection)
    """
    # Branch 1: 1x1 conv
    branch1 = Conv2D(filters_1x1, (1, 1), padding='same', activation='relu',
                      name=f'{name}_1x1')(x)

    # Branch 2: 1x1 reduce -> 3x3 conv
    branch2 = Conv2D(filters_3x3_reduce, (1, 1), padding='same', activation='relu',
                      name=f'{name}_3x3_reduce')(x)
    branch2 = Conv2D(filters_3x3, (3, 3), padding='same', activation='relu',
                      name=f'{name}_3x3')(branch2)

    # Branch 3: 1x1 reduce -> 5x5 conv
    branch3 = Conv2D(filters_5x5_reduce, (1, 1), padding='same', activation='relu',
                      name=f'{name}_5x5_reduce')(x)
    branch3 = Conv2D(filters_5x5, (5, 5), padding='same', activation='relu',
                      name=f'{name}_5x5')(branch3)

    # Branch 4: 3x3 max-pool -> 1x1 projection
    branch4 = MaxPooling2D((3, 3), strides=(1, 1), padding='same',
                            name=f'{name}_pool')(x)
    branch4 = Conv2D(filters_pool_proj, (1, 1), padding='same', activation='relu',
                      name=f'{name}_pool_proj')(branch4)

    # Concatenate all branches along the channel axis
    return Concatenate(axis=-1, name=f'{name}_concat')([branch1, branch2, branch3, branch4])


# Define the input shape
input_shape = (224, 224, 3)
inputs = Input(shape=input_shape)

# Simple stem before the Inception block (mirrors the ResNet cell's stem above)
x = Conv2D(64, (7, 7), strides=(2, 2), padding='same', activation='relu')(inputs)
x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

# One Inception module (filter counts taken from GoogLeNet's 3a block)
x = inception_module(x,
                      filters_1x1=64,
                      filters_3x3_reduce=96, filters_3x3=128,
                      filters_5x5_reduce=16, filters_5x5=32,
                      filters_pool_proj=32,
                      name='inception_3a')

# Classification head
x = GlobalAveragePooling2D()(x)
outputs = Dense(4, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


### Why the channel counts line up

All four branches use `padding='same'` and no spatial stride, so every branch outputs the same height/width as the input to the block — only the channel depth differs per branch (64 + 128 + 32 + 32 = 256 channels here). `Concatenate` requires matching spatial dimensions, which is exactly why `padding='same'` is non-negotiable in every branch.

## 5. Bonus: Depthwise Separable Convolutions (MobileNet-style)

A fourth pattern worth knowing alongside plain conv, residual, and Inception blocks: MobileNet's depthwise separable convolution splits a standard convolution into (1) a **depthwise** conv that filters each input channel independently, followed by (2) a **pointwise** (1x1) conv that mixes channels. This cuts parameters and FLOPs roughly by a factor of the kernel size squared, which matters a lot if you're ever deploying a model on constrained hardware (e.g. a bedside monitor or embedded device) rather than a GPU server.

In [ ]:
from tensorflow.keras.layers import Input, DepthwiseConv2D, Conv2D, BatchNormalization, Activation, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model


def depthwise_separable_block(x, pointwise_filters, stride=1, name='ds_block'):
    x = DepthwiseConv2D((3, 3), strides=stride, padding='same', name=f'{name}_dwconv')(x)
    x = BatchNormalization(name=f'{name}_dw_bn')(x)
    x = Activation('relu', name=f'{name}_dw_act')(x)

    x = Conv2D(pointwise_filters, (1, 1), padding='same', name=f'{name}_pwconv')(x)
    x = BatchNormalization(name=f'{name}_pw_bn')(x)
    x = Activation('relu', name=f'{name}_pw_act')(x)
    return x


inputs = Input(shape=(224, 224, 3))
x = Conv2D(32, (3, 3), strides=2, padding='same', activation='relu')(inputs)
x = depthwise_separable_block(x, pointwise_filters=64)
x = depthwise_separable_block(x, pointwise_filters=128, stride=2)
x = GlobalAveragePooling2D()(x)
outputs = Dense(4, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 6. Summary: When to Reach for Which API

| Architecture pattern | Needs Functional API? | Why |
|---|---|---|
| Plain stack (Section 1–2) | No | Single chain, no branching |
| Residual block (Section 3) | **Yes** | `Add()` needs a reference to an earlier tensor |
| Inception block (Section 4) | **Yes** | Parallel branches on one input, rejoined via `Concatenate` |
| Depthwise separable (Section 5) | No (but often used inside larger Functional models) | Still a chain by itself |

**Rule of thumb:** if you can draw the architecture as a single line of boxes, `Sequential` is fine and more concise. The moment a tensor needs to be reused, split, or merged with another tensor, switch to the `Functional` API (or `Model` subclassing for anything with control flow).

## 7. Exercise 

Using the `inception_module` function above as a template, implement a **DenseNet-style dense block**: instead of concatenating parallel branches of the *same* input, each new conv layer should take as input the **concatenation of all previous layers' outputs** in the block (not just the block's original input). Stack 3 conv layers this way, each producing `growth_rate=32` new feature maps, and print the final channel count leaving the block.

In [ ]:
# Your implementation here
# Hint: keep a running list of feature maps, e.g.
#   features = [x]
#   for i in range(num_layers):
#       merged = Concatenate()(features) if len(features) > 1 else features[0]
#       new_features = Conv2D(growth_rate, (3, 3), padding='same', activation='relu')(merged)
#       features.append(new_features)
#   x = Concatenate()(features)
